# rust-NMF — comprehensive tutorial

End-to-end walk-through on omicverse `pbmc8k` (7750 cells × 20939
genes). Covers:

1. **R baseline** — `NMF::nmf()` reference timing and result, used to
   anchor every subsequent measurement.
2. **Bit-equivalent Rust replacement** — same V/W₀/H₀, same iterates,
   max abs diff < 1e-12. Use this when you need to reproduce an
   existing R analysis exactly, but faster.
3. **All five built-in algorithms** — `brunet`, `lee`, `offset`,
   `nsNMF`, `hals` — what each one minimises and when to pick it.
4. **Initialisation** — random vs NNDSVD; impact on iterations and
   factor identity.
5. **Stopping criteria** — fixed `max_iter` vs `stationary` (R-parity).
6. **Threading** — per-call rayon pools (`num_threads=N`).
7. **The production recipe** — HALS + NNDSVD + 25 iters: ~200× R.
8. **Inspecting factors** — top genes, cell-type composition.

Each cell is a single experiment so you can copy-paste the bits you
need. Outputs in this notebook were generated by re-executing on
Sherlock with 16 cores.

## 0. Setup

Imports and a couple of helpers we'll reuse.

In [ ]:
import os, subprocess, time
from pathlib import Path
import numpy as np, pandas as pd
import scanpy as sc, anndata as ad
import matplotlib.pyplot as plt
from scipy.optimize import linear_sum_assignment
import nmf_rs

PBMC = '/scratch/users/steorra/analysis/omicverse_dev/omicverse/data/pbmc8k.h5ad'
RSCRIPT = '/scratch/users/steorra/env/CMAP/bin/Rscript'
RLIB    = '/scratch/users/steorra/env/CMAP_Rlib'
PATH_R  = '/scratch/users/steorra/env/CMAP/bin'
WORK = Path('tutorial_work'); WORK.mkdir(exist_ok=True)
HVG_N, RANK = 2000, 10

In [ ]:
def reconstruction_loss(V, W, H):
    return 0.5 * float(np.linalg.norm(V - W @ H) ** 2)

def hungarian_match(A, B):
    """Match rows of B to rows of A by max Pearson; returns matched corr."""
    n = A.shape[0]
    corr = np.corrcoef(A, B)[:n, n:]
    row, col = linear_sum_assignment(-corr)
    return corr[row, col], row, col

### Load + preprocess pbmc8k

Standard scanpy log-normalisation + top-2000 HVGs (Seurat v3).
We use a **dense** `V` of shape (genes × cells) because R's NMF
needs that. For very large datasets, see the `examples/` notes on
future sparse support.

In [ ]:
ad_orig = ad.read_h5ad(PBMC)
ad_orig.layers['counts'] = ad_orig.X.copy()
sc.pp.normalize_total(ad_orig, target_sum=1e4)
sc.pp.log1p(ad_orig)
sc.pp.highly_variable_genes(ad_orig, n_top_genes=HVG_N,
                            flavor='seurat_v3', layer='counts')
ad_hvg = ad_orig[:, ad_orig.var.highly_variable].copy()
V = np.ascontiguousarray(ad_hvg.X.toarray().T.astype(np.float64))
n_genes, n_cells = V.shape
print(f'V (genes × cells) = {V.shape}   non-negative? {bool((V>=0).all())}')

## 1. R `NMF::nmf()` baseline

We seed `W0`/`H0` in R via `set.seed(2024); runif(...)` so that the
same starting point is reusable from Rust later.

We time both `lee` (Frobenius) and `brunet` (KL) for 100 iterations.

In [ ]:
def write_input_tsvs():
    np.savetxt(WORK/'V.tsv', V, delimiter='\t')
    rscript = (
        'set.seed(2024L)\n'
        "V <- as.matrix(read.table('{0}/V.tsv', sep='\\t'))\n"
        'W0 <- matrix(runif({1}*{2}, 0, max(V)), {1}, {2})\n'
        'H0 <- matrix(runif({2}*{3}, 0, max(V)), {2}, {3})\n'
        "write.table(W0, '{0}/W0.tsv', sep='\\t', row.names=FALSE, col.names=FALSE)\n"
        "write.table(H0, '{0}/H0.tsv', sep='\\t', row.names=FALSE, col.names=FALSE)\n"
    ).format(WORK, n_genes, RANK, n_cells)
    (WORK/'seed.R').write_text(rscript)
    subprocess.run([RSCRIPT, str(WORK/'seed.R')], check=True,
                   env={**os.environ, 'PATH': PATH_R + ':' + os.environ.get('PATH','')})
write_input_tsvs()
W0 = np.loadtxt(WORK/'W0.tsv'); H0 = np.loadtxt(WORK/'H0.tsv')
W0.shape, H0.shape

In [ ]:
rscript = ('''
suppressPackageStartupMessages({{
  .libPaths(c("{rlib}", .libPaths()))
  library(NMF)
}})
args   <- commandArgs(trailingOnly=TRUE); method <- args[1]
V      <- as.matrix(read.table("{w}/V.tsv",  sep="\\t"))
W0     <- as.matrix(read.table("{w}/W0.tsv", sep="\\t"))
H0     <- as.matrix(read.table("{w}/H0.tsv", sep="\\t"))
.seed  <- function(model, target, ...) {{ basis(model)<-W0; coef(model)<-H0; model }}
t0     <- proc.time()[3]
fit    <- nmf(V, rank=ncol(W0), method=method, seed=.seed,
              .options="-cb", .pbackend=NA, nrun=1, maxIter=100,
              stopconv=10000L)
cat(sprintf("R_TIME %s %.4f\\n", method, proc.time()[3] - t0))
write.table(basis(fit), sprintf("{w}/%s_W_R.tsv", method),
            sep="\\t", row.names=FALSE, col.names=FALSE)
write.table(coef(fit),  sprintf("{w}/%s_H_R.tsv", method),
            sep="\\t", row.names=FALSE, col.names=FALSE)
''').format(rlib=RLIB, w=WORK)
(WORK/'run_R.R').write_text(rscript)
r_times = {}
for method in ('lee', 'brunet'):
    out = subprocess.run([RSCRIPT, str(WORK/'run_R.R'), method],
                         capture_output=True, text=True,
                         env={**os.environ, 'PATH': PATH_R + ':' + os.environ.get('PATH','')})
    for line in out.stdout.splitlines():
        if line.startswith('R_TIME'):
            _, m, t = line.split(); r_times[m] = float(t)
    print(f'R {method:6s} 100 iters: {r_times[method]:.2f} s')

These are the numbers everything else gets compared against.

## 2. Bit-equivalent Rust replacement

Same `V`, `W0`, `H0`, same algorithm, same `max_iter` — output is
**bitwise identical** to R within f64 round-off (~1e-12). Use this
to reproduce a published R analysis without changing results.

In [ ]:
def reproduce(method, max_iter=100, threads=16):
    t = time.perf_counter()
    res = nmf_rs.nmf(V, rank=RANK, method=method,
                     W0=W0.copy(), H0=H0.copy(),
                     max_iter=max_iter, stop='max_iter',
                     num_threads=threads)
    dt = time.perf_counter() - t
    return res, dt

rows = []
for method in ('lee', 'brunet'):
    res, dt = reproduce(method)
    W_R = np.loadtxt(WORK/f'{method}_W_R.tsv')
    H_R = np.loadtxt(WORK/f'{method}_H_R.tsv')
    rows.append({
        'method':       method,
        'R (s)':        r_times[method],
        'Rust 16t (s)': dt,
        'speed-up':     r_times[method] / dt,
        'max|ΔW|':     float(np.abs(res.W - W_R).max()),
        'max|ΔH|':     float(np.abs(res.H - H_R).max()),
    })
pd.DataFrame(rows).round(3)

max abs diffs are at f64 round-off (~1e-12) — *bitwise* equal to R.
Use this when you must reproduce an existing analysis.

## 3. The five built-in algorithms

Each minimises a different objective and has a different update
structure. Same `nmf()` entry-point — only the `method` argument
changes.

| `method`     | Objective                        | When to use it |
|--------------|----------------------------------|----------------|
| `'brunet'`   | KL divergence                    | Default in R; good for count-like data |
| `'lee'`      | Frobenius / Euclidean            | Most common in single-cell / sklearn |
| `'offset'`   | Frobenius + per-feature baseline | Removes a learned background per gene |
| `'nsNMF'`    | KL with smoothing (Brunet + S)   | Sparser, more interpretable factors |
| `'hals'`     | Frobenius (block-coord least-sq) | Same loss as Lee, **5–10× fewer iters** |

In [ ]:
rows = []
for method in ('brunet', 'lee', 'offset', 'nsNMF', 'hals'):
    res, dt = reproduce(method, max_iter=100)
    rows.append({
        'method': method,
        'time 16t (s)': dt,
        'iters': res.n_iter,
        'loss': reconstruction_loss(V, res.W, res.H),
    })
pd.DataFrame(rows).round(3)

Same `(V, W0, H0)`, 100 iterations each. `hals` reaches the lowest
loss in this comparison; `nsNMF` is highest (it adds a smoothing
regulariser that trades off pure reconstruction for sparsity).

## 4. Initialisation: random vs NNDSVD

Random `runif(0, max(V))` puts `||W₀ H₀||` orders of magnitude above
`||V||`, so the first ~10 iterations are spent shrinking the factors.
**NNDSVD** seeds with the non-negative projection of the truncated
SVD — the initial loss is already close to the converged plateau.

In [ ]:
t = time.perf_counter()
W0_nn, H0_nn = nmf_rs.nndsvd_init(V, RANK, fill='mean', seed=0)
print(f'nndsvd_init: {time.perf_counter()-t:.2f} s')

print(f'||V||²/2          = {0.5*np.linalg.norm(V)**2:.3e}')
print(f'loss at runif W0H0 = {reconstruction_loss(V, W0,    H0   ):.3e}')
print(f'loss at NNDSVD W0H0= {reconstruction_loss(V, W0_nn, H0_nn):.3e}')

NNDSVD's starting loss is **5 orders of magnitude** smaller than
random's. That budget gets paid back as faster convergence.

In [ ]:
rows = []
for init_label, (Wi, Hi) in [('random', (W0, H0)), ('NNDSVD', (W0_nn, H0_nn))]:
    for max_it in (10, 25, 50, 100):
        t = time.perf_counter()
        res = nmf_rs.nmf(V, rank=RANK, method='lee',
                         W0=Wi.copy(), H0=Hi.copy(),
                         max_iter=max_it, stop='max_iter', num_threads=16)
        rows.append({'init': init_label, 'iters': max_it,
                     'time (s)': time.perf_counter()-t,
                     'loss': reconstruction_loss(V, res.W, res.H)})
pd.DataFrame(rows).pivot(index='iters', columns='init',
                         values=['time (s)', 'loss']).round(3)

Same algorithm (`lee`), only the init differs. NNDSVD reaches at
10 iters what random reaches at ~50.

## 5. Stopping criteria

`stop='max_iter'` runs exactly `max_iter` iterations (the default).
`stop='stationary'` replicates R's `nmf.stop.stationary` semantics —
halt when the objective is flat over a window of `check_niter` checks
taken every `check_interval` iterations. Returns the **deviance
trace** in `res.deviances`.

In [ ]:
res = nmf_rs.nmf(V, rank=RANK, method='lee',
                 W0=W0_nn.copy(), H0=H0_nn.copy(),
                 max_iter=2000, stop='stationary',
                 stationary_th=1e-3, check_interval=10, check_niter=10,
                 num_threads=16)
print(f'stopped at iter {res.n_iter}, final loss = {res.deviances[-1]:.3e}')

fig, ax = plt.subplots(figsize=(5.5, 3.0))
ax.plot(res.deviances, lw=1.5)
ax.set_yscale('log')
ax.set_xlabel('check #'); ax.set_ylabel('Frobenius loss')
ax.set_title('lee + NNDSVD, stop="stationary" (th=1e-3)')
fig.tight_layout()

## 6. Per-call thread pools (`num_threads=`)

`num_threads=N` builds a fresh rayon worker pool scoped to *this one*
`nmf()` call. Different calls in the same process can use different
thread counts without re-initialising a global pool.

In [ ]:
rows = []
for nt in (1, 2, 4, 8, 16):
    t = time.perf_counter()
    _ = nmf_rs.nmf(V, rank=RANK, method='lee',
                   W0=W0.copy(), H0=H0.copy(),
                   max_iter=100, num_threads=nt)
    rows.append({'threads': nt,
                 'lee 100it (s)': time.perf_counter() - t})
pd.DataFrame(rows).set_index('threads').round(3)

## 7. The production recipe: HALS + NNDSVD

If you don't need bit-equivalence with R (you just need a good
non-negative factorisation, fast), this is the fastest configuration:

```python
W0, H0 = nmf_rs.nndsvd_init(V, rank)
res    = nmf_rs.nmf(V, rank, method='hals',
                    W0=W0, H0=H0, max_iter=25, num_threads=16)
```

In [ ]:
configs = [
    ('R lee 100',        None,   None,    100),
    ('rust lee + rand',  'lee',  'rand',  100),
    ('rust lee + NNDSVD','lee',  'nn',     50),
    ('rust hals + rand', 'hals', 'rand',   50),
    ('rust hals + NNDSVD','hals','nn',     25),
    ('rust hals + NNDSVD (loose)', 'hals', 'nn', 10),
]
rows = []
for label, method, init, mit in configs:
    if method is None:
        rows.append({'config': label, 'time (s)': r_times['lee'],
                     'iters': 100, 'loss': None,
                     'speed-up vs R lee': 1.0})
        continue
    Wi, Hi = (W0, H0) if init == 'rand' else (W0_nn, H0_nn)
    t = time.perf_counter()
    res = nmf_rs.nmf(V, rank=RANK, method=method,
                     W0=Wi.copy(), H0=Hi.copy(),
                     max_iter=mit, stop='max_iter', num_threads=16)
    dt = time.perf_counter() - t
    rows.append({'config': label, 'time (s)': dt, 'iters': res.n_iter,
                 'loss': reconstruction_loss(V, res.W, res.H),
                 'speed-up vs R lee': r_times['lee'] / dt})
pd.DataFrame(rows).round(3)

**HALS+NNDSVD@25 ≈ 200× R**, same loss plateau. The 10-iter loose
variant gives ~280× R for cases where you just need a quick first
factorisation pass.

## 8. How similar are HALS factors to R Lee's?

Hungarian-match HALS+NNDSVD@25 columns to R lee@100, report
Pearson on `H` and `W` per matched pair.

In [ ]:
res_hals = nmf_rs.nmf(V, rank=RANK, method='hals',
                      W0=W0_nn.copy(), H0=H0_nn.copy(),
                      max_iter=25, num_threads=16)
H_R = np.loadtxt(WORK/'lee_H_R.tsv')
W_R = np.loadtxt(WORK/'lee_W_R.tsv')
matched_h, row, col = hungarian_match(H_R, res_hals.H)
matched_w = np.array([float(np.corrcoef(W_R[:, r], res_hals.W[:, c])[0, 1])
                       for r, c in zip(row, col)])
df = pd.DataFrame({'r_H (matched)': matched_h, 'r_W (matched)': matched_w},
                   index=[f'factor {i}' for i in range(RANK)])
df.loc['mean'] = df.mean()
df.round(3)

8–9 of 10 factors match R Lee at r > 0.85 (often > 0.95). The
occasional 1–2 mismatches are NMF's local-minimum spread — expected
for any non-bit-equivalent solver. **If you need 1.0 correlation
with R, use `method='lee'` or `'brunet'` instead** (Section 2).

## 9. Single-step kernels (advanced)

If you want full control over the iteration loop — e.g. interleave
with a custom regulariser — the per-step kernels are exposed:

In [ ]:
H1 = nmf_rs.update_h_brunet(V, W0, H0)
W1 = nmf_rs.update_w_brunet(V, W0, H1)
print('one Brunet step shapes:', H1.shape, W1.shape)

H1 = nmf_rs.update_h_lee(V, W0, H0)
W1 = nmf_rs.update_w_lee(V, W0, H1)
print('one Lee step shapes:', H1.shape, W1.shape)

## 10. Factor interpretation

Top-loaded genes per factor in `W` and the cell-type composition of
the most-active cells per factor in `H`.

In [ ]:
gene_names = np.array(ad_hvg.var_names)
top = pd.DataFrame({
    f'factor {k}': gene_names[np.argsort(-res_hals.W[:, k])[:5]]
    for k in range(RANK)
})
top

In [ ]:
ct_col = ('predicted_celltype' if 'predicted_celltype' in ad_hvg.obs.columns
          else 'cell_type')
comp = pd.DataFrame({
    f'factor {k}': ad_hvg.obs.iloc[np.argsort(-res_hals.H[k, :])[:50]]
                          [ct_col].value_counts().head(3)
    for k in range(RANK)
})
comp.fillna(0).astype(int)

Each factor concentrates in 1–2 cell types — exactly what NMF is
supposed to do for biological program discovery.

## Cleanup

In [ ]:
import shutil
shutil.rmtree(WORK, ignore_errors=True)